# Tool Calling

**Prerequisites:** `01_your_first_llm.ipynb`, `05_structured_outputs.ipynb`

Not agents — just the one round trip every agent is built from:

```
LLM  ->  tool request  ->  Python executes  ->  tool result  ->  LLM
```

No loop, no planning, no deciding what to do next. `02_agent_runtime` and `03_tools` build on exactly this mechanic — this notebook is where it comes from.

**By the end you'll have:**
- Declared tool schemas and watched the model choose one, with real arguments
- Seen the model correctly *not* call a tool when none is needed
- Completed a full round trip: execute, feed the result back, get the final answer
- Inspected Groq's real structured error object and understood exactly why an API-level rejection is called that
- Split tool-calling failure into two real categories — an API-level rejection Groq already handles for tool *existence*, and an argument validation failure that's genuinely ours to catch, Pydantic-backed, covering both type and domain-specific rules
- Built `execute_tool_call()` and `resolve_tools()` (with retry-on-failure) in `llm.py`

## Tool schema

A tool is a JSON schema describing a function's name, purpose, and parameters — the model never runs any code, it only ever produces a request to call one. Two tools, and the real Python functions behind them:

## Function calling

Pass `tools` to the raw client (not the `chat()` wrapper — it only returns text, and the interesting part of this response, `tool_calls`, is a different field on the message that `chat()` throws away):

`content` is `None` — the model didn't answer, it asked to call `multiply` with `{"a": 918273, "b": 46571}`. It chose `multiply` over `add` from nothing but the two tools' `description` fields, and the arguments are real numbers, not strings.

**Why such awkward numbers?** This example used to ask for `12 times 7`, and a capable model simply answers it: `12 × 7 = 84`, `tool_calls: None`. That is not a failure — it is the model correctly judging that a tool it could do in its head is not worth calling. But it leaves nothing to demonstrate, and every cell below here depends on a tool call existing.

Six-digit multiplication is different: getting it wrong is easy and the tool is exactly right, so the model reaches for it without being told to. That is the behaviour worth seeing — a tool is offered, and the model decides. Chapter 2 leans on the same judgment when it decides whether to delegate work to a sub-agent.

A question that doesn't need either tool gets answered directly, with no tool call at all — worth noting this isn't guaranteed for *every* off-topic question. "Tell me a joke" turned out to be a bad test case here: across repeated runs it sometimes made up a pun involving `add`, treating the tools as material to work with rather than ignoring them. "What color is the sky?" is unambiguous enough to stay clean (using different variable names here on purpose, so `message` above still holds the tool-call response for the next section):

## Arguments

`tool_call.function.arguments` is a plain JSON string — it needs parsing before you can use it, same as any other model output:

## Completing the round trip

Execute the tool, send the result back as a `role="tool"` message tied to the original `tool_call.id`, and call again for the final natural-language answer:

The name-lookup-and-execute step (`FUNCTIONS[tool_call.function.name](**args)`) is worth wrapping the same way `Conversation` and `extract()` were — with a basic guard against your own `tools` schema and `functions` dict falling out of sync, so a typo'd or half-wired-up tool raises a clear error instead of crashing on a `KeyError`:

```python
def execute_tool_call(tool_call, functions):
    name = tool_call.function.name
    if name not in functions:
        raise ValueError(f"no tool named '{name}' is available")
    args = json.loads(tool_call.function.arguments)
    result = functions[name](**args)
    return str(result)
```

## Validation

Two different things can go wrong, and both are real, not hypothetical:

**The model can request a tool that was never declared.** `add` and `multiply` were the only tools offered, but for an unrelated factual question the model tried to call `brave_search` — a tool it apparently expects to have from its own training, not one we gave it.

Worth being precise about why this is called an **API failure**, since "the tool" here is nothing more than a local Python function — the API being referred to is *Groq's* Chat Completions endpoint, the one `client.chat.completions.create(...)` sends an HTTPS request to. A normal call gets back HTTP 200 and a completion. Here, Groq's own server checks the tool name the model tried to use against the `tools` list *we* sent in the request, finds `brave_search` isn't in it, and refuses to complete the request at all — sending back HTTP 400 instead, which the SDK raises as `BadRequestError`. We never get a `message` or `tool_call` object; the failure happens at the level of the request itself, before there's anything of ours to inspect. That's the contrast with category 2 later: there, the request succeeds (200 OK, a real object comes back) but *that object* fails a check on our side.

Two things worth knowing about `e.body`, the actual structured error Groq sends:

- **`message`** is a plain sentence describing what went wrong — here, "you tried to call a tool that isn't in your `tools` list." Safe to read, safe to reuse.
- **`failed_generation`** is the literal text the model tried to generate — the mistake itself, verbatim, in whatever broken syntax it came out as.

And the bigger realization from `message` alone: it already confirms **whether a tool exists is Groq's problem to validate, not ours.** By the time we ever see a real `tool_call` object, its name is guaranteed to be one of the names in the `tools` list we sent — there's no need to re-check that ourselves.

What Groq does *not* check is whether the *arguments* make sense — neither their basic type, nor any domain-specific rule a JSON schema can't express (like "the divisor can't be zero"). That part genuinely is our job, and it's exactly what Pydantic is for — the same tool notebook 5 used for structured output, applied here to tool arguments instead.

That error is completely safe to send back to the model, and it's worth noticing exactly why: unlike the `brave_search` 400 above (which contains the model's own malformed generated text), this `ValidationError` repeats nothing the model wrote. It's a description *we* generated, about *our own* schema — "`b` should be a number, you sent a string." Safe to echo, and genuinely useful feedback for a retry, unlike the earlier case.

`execute_tool_call()` runs exactly this check, whenever it's given a `schemas` argument (backward compatible — omit it and you get the old, unchecked behavior). It still keeps a basic `name not in functions` guard too — not because the model might hallucinate past what we already established Groq blocks, but as a cheap safety net against *our own* mistake of declaring a tool without ever wiring up its implementation:

```python
def execute_tool_call(tool_call, functions, schemas=None):
    name = tool_call.function.name
    if name not in functions:
        raise ValueError(f"no tool named '{name}' is available")
    args = json.loads(tool_call.function.arguments)
    if schemas is not None:
        args = schemas[name].model_validate(args).model_dump()
    result = functions[name](**args)
    return str(result)
```

## Retries

Detecting a problem isn't the same as recovering from it. Notebook 5's `extract()` didn't just spot a bad response, it fed the error back and tried again — this should too, but the two categories need genuinely different handling, in both *what* gets said and *how* it's delivered:

1. **API-level rejection** (the hallucinated `brave_search` case) — there's no `tool_call.id` to tie a result to, since the API never returned one, so the retry has to be a plain `user` message. And here's the part worth being exact about: Groq's error *does* contain a safe part (`e.body["message"]`, shown above) alongside the risky part (`failed_generation`) — this isn't a case of "nothing safe exists." We're choosing not to split them here. Reliably picking `message` apart from `failed_generation` means knowing Groq's exact error shape, and a different provider might not structure its errors the same way at all — that per-provider parsing is exactly the kind of thing `03_tools` should own properly. For now, the retry below is a deliberately generic nudge, not because nothing safe exists, but because extracting it isn't built yet.
2. **Argument validation failure** (wrong type, or a valid type that still breaks a domain rule — from `execute_tool_call`'s `schemas` check) — there *is* a real `tool_call.id` this time, so the error goes back as that tool's own `role="tool"` result — and because the error is ours, not the model's, it's safe to make it the *actual* reason, exactly like `extract()` does.

Both still count against the same retry limit.

`0` is a completely ordinary, valid float — no type check catches it. The `@field_validator` is what catches it, and it does so *before* `divide()` ever runs: the actual Python function never executes with `b=0`, never raises a real `ZeroDivisionError`. That's the point of validating first — the bad value is stopped upstream, not caught after the fact.

Unlike the earlier type error, though, there's something genuinely different about this failure: there's no "corrected" `b` that both fixes the validation error *and* still answers "what is 10 divided by 0?" The question itself has no valid tool call that satisfies it. Worth seeing what that means for retrying — `resolve_tools()` in `llm.py`, now accepting an optional `schemas` for exactly this kind of check:

```python
def resolve_tools(messages, tools, functions, schemas=None, retries=3, model=DEFAULT_MODEL):
    messages = list(messages)
    for _ in range(retries):
        try:
            response = client.chat.completions.create(model=model, messages=messages, tools=tools)
        except BadRequestError:
            messages.append({
                "role": "user",
                "content": "That tool call could not be processed. Try again, calling one of the available tools correctly.",
            })
            continue

        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            try:
                content = execute_tool_call(tool_call, functions, schemas)
            except Exception as e:
                content = f"Error: {e}"
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": content})

    raise ValueError(f"Failed to get a final answer after {retries} attempts")
```

Starting from a conversation where the model already tried `divide(10, 0)` once (simulated by hand — live, the model usually either reasons about this in plain text or hits an unrelated generation glitch before ever producing a clean `divide(10, 0)` call, so this isn't reliably reproducible on demand):

It resolves, but notice *how* — the model doesn't retry with some other value of `b`, because no other value answers the original question. It recognizes the operation itself is impossible and explains that in plain text instead. That's the real distinction from a type error: when a fix exists, the specific validation error lets the model find it directly; when no fix exists, the same specific error still does its job by making the impossibility clear enough for the model to give up on the tool and just answer honestly, rather than looping on bad arguments.

Given the exact reason `b` was invalid, the model corrects `"three"` to `3` on the very next call — no vague "try again," a specific fix for the specific problem. That's the payoff of category 2 being safe to echo: the retry can be genuinely informative instead of a generic nudge.

## What you built

✓ Declared tool schemas and watched the model pick the right one with real, typed arguments

✓ Confirmed the model correctly skips tool use when no tool is needed

✓ Completed a full call → execute → feed-back → final-answer round trip

✓ Inspected Groq's real structured error (`e.body`) and split it into a safe part (`message`) and a risky part (`failed_generation`, the model's own broken syntax) — and understood *why* it's called an API failure: Groq's own Chat Completions endpoint rejects the request outright, before any `tool_call` object exists to inspect

✓ Split failure into two genuinely different categories: an API-level rejection (a safe part of the error exists, but extracting it reliably is provider-specific work deferred to `03_tools`, so the retry stays a deliberate generic nudge for now) and an argument validation failure (wrong type, or a valid type that still breaks a domain rule — always safe to describe exactly, since it's our own schema talking, not the model's broken syntax)

✓ Built `execute_tool_call()` (with optional Pydantic `schemas` validation, including custom `field_validator`s for domain rules) and `resolve_tools()` in `llm.py` — recovering from both categories by feeding back a generic nudge or the real reason, whichever is currently safe

✓ Saw the two categories recover differently: a type error gets corrected on the very next attempt; an impossible operation (divide by zero) has no argument fix at all, so recovery means the model abandoning the tool and answering in text instead

This is deliberately *not* an open-ended loop — `resolve_tools` has a fixed retry budget for recovering from failure, not a policy for deciding what to do next. That decision, along with weaving tool calls and structured extraction into an ongoing `Conversation`, is `02_agent_runtime`'s job — and `09_build_an_agent` is where `Conversation` itself grows to support this. A reusable, general-purpose tools framework built on this exact mechanic is `03_tools`'s.

**Next:** `07_streaming.ipynb` — streaming tokens and events.